# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [12]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# TODO: Load environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY,
    model_name="text-embedding-ada-002",
    api_base="https://openai.vocareum.com/v1"
)

In [7]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
        
    )

In [9]:
# Demonstrate semantic search from ChromaDB
print("=" * 60)
print("SEMANTIC QUERY DEMO")
print("=" * 60)

test_queries = [
    "racing game for PlayStation",
    "Mario platformer Nintendo",
    "fighting game"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 40)
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    for i, doc in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        print(f"  Result {i+1}:")
        print(f"    Name: {meta.get('Name')}")
        print(f"    Platform: {meta.get('Platform')}")
        print(f"    Year: {meta.get('YearOfRelease')}")
        print(f"    Description: {doc[:100]}...")

SEMANTIC QUERY DEMO

Query: 'racing game for PlayStation'
----------------------------------------
  Result 1:
    Name: Gran Turismo 5
    Platform: PlayStation 3
    Year: 2010
    Description: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection ...
  Result 2:
    Name: Gran Turismo
    Platform: PlayStation 1
    Year: 1997
    Description: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars an...
  Result 3:
    Name: Grand Theft Auto: San Andreas
    Platform: PlayStation 2
    Year: 2004
    Description: [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the ficti...

Query: 'Mario platformer Nintendo'
----------------------------------------
  Result 1:
    Name: Super Mario 64
    Platform: Nintendo 64
    Year: 1996
    Description: [Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the ...
  Result 